# Configured ROBERT forward model

This notebook validates a schema-version-2 configuration, optionally prepares its opacity cache, evaluates the configured emission or transmission model, inspects the portable NPZ product, and plots every selected dataset. Run it with the `robert-exoplanets` Conda kernel.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from robert_exoplanets.io.configured_tasks import (
    describe_config,
    load_observations,
    prepare_opacity,
    run_forward_task,
)
from robert_exoplanets.io.task_config import (
    initialize_task_directories,
    load_task_config,
)

repository = Path.cwd().resolve()
if not (repository / "pyproject.toml").is_file():
    repository = Path.cwd().resolve().parents[1]
assert (repository / "pyproject.toml").is_file(), "Start Jupyter in the ROBERT repository or notebook directory"
repository

## Select and validate the YAML

Copy a supplied configuration before changing it. Relative paths are resolved from the YAML, not from the notebook. Validation does not load observations, opacity, FastChem, optical constants, or PHOENIX files.

In [ ]:
CONFIG_PATH = repository / "configurations" / "quickstart.yaml"

config = load_task_config(CONFIG_PATH)
print(describe_config(config))

In [ ]:
print("Observations:", config.observations.path)
print("Opacity source:", config.opacity.path)
print("Opacity cache:", config.opacity.cache_directory)
print("Outputs:", config.outputs.directory)
print("Scratch:", config.runtime.scratch_directory)
print("Datasets:", config.observations.datasets)
print("Forward values:")
for parameter in config.parameters:
    print(f"  {parameter.name}: {parameter.value if parameter.value is not None else 'prior midpoint'}")

## Prepare opacity

Set `PREPARE_OPACITY = True` after the observation and opacity paths above are valid. Preparation is needed once for a new source opacity, wavelength binning, species set, or quadrature configuration. Run this cell on one process.

In [ ]:
PREPARE_OPACITY = False

initialize_task_directories(config)
if PREPARE_OPACITY:
    observations = load_observations(config)
    prepare_opacity(config, observations)
    print("Opacity preparation complete")
else:
    print("Opacity preparation skipped; set PREPARE_OPACITY = True when needed")

## Evaluate the forward model

Set `RUN_MODEL = True` after the cache and all external inputs are available. `run_forward_task` constructs the complete configured problem, uses each explicit parameter `value` or its prior midpoint, evaluates all datasets, and writes `outputs/forward_model.npz`.

In [ ]:
RUN_MODEL = False

forward_path = config.outputs.directory / "forward_model.npz"
if RUN_MODEL:
    forward_path = run_forward_task(config, CONFIG_PATH)
    print("Wrote", forward_path)
elif forward_path.is_file():
    print("Using existing", forward_path)
else:
    print("No product yet; set RUN_MODEL = True")

## Inspect and plot the product

The archive has one wavelength/model pair per selected dataset and one scalar per forward parameter. The modeled values are dimensionless eclipse depth for emission or transit depth for transmission.

In [ ]:
if forward_path.is_file():
    with np.load(forward_path, allow_pickle=False) as archive:
        print("Archive keys:")
        print("\n".join(f"  {key}" for key in archive.files))
        forward_arrays = {key: np.array(archive[key], copy=True) for key in archive.files}
else:
    forward_arrays = {}

forward_parameters = {
    key.removeprefix("parameter_"): float(value)
    for key, value in forward_arrays.items()
    if key.startswith("parameter_")
}
forward_parameters

In [ ]:
if forward_arrays:
    figure, axis = plt.subplots(figsize=(10, 5.5))
    for dataset in config.observations.datasets:
        wavelength = forward_arrays[f"{dataset}_wavelength_micron"]
        spectrum = forward_arrays[f"{dataset}_model"]
        axis.plot(wavelength, 1e6 * spectrum, marker="o", label=dataset)
    axis.set_xlabel("Wavelength (micron)")
    axis.set_ylabel("Observable (ppm)")
    axis.set_title(config.run.name)
    axis.legend()
    figure.tight_layout()
    plt.show()
else:
    print("Run the model before plotting")

## Regenerate ROBERT diagnostics

When `plotting.enabled` and `plotting.forward` are true, the forward task writes its diagnostics automatically. They can also be regenerated from the saved NPZ with:

```bash
python postprocess_forward.py --config configuration.yaml
```

This creates fit statistics, the exact fixed-parameter mapping, a spectrum/residual plot, and a plot manifest under `outputs/plots/forward/`.